# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202407_Hurricane_Beryl'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'tropomi'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 13 .tif files in the S3 bucket.


['drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0702.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0703.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0704.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0705.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0706.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0707.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0708.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0709.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0710.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0703.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0704.tif',
 'drcs_activations/202407_Hurricane_Beryl/trop

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 150
  - Total size: 12.38 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023126133547_aid0001.tif (1.6 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043620_aid0001.tif (0.5 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023127043712_aid0001.tif (12.1 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131025921_aid0001.tif (0.4 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023131030013_aid0001.tif (28.9 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRESS/ControlData_May2023/May23_context/ECO2LSTE.001_SDS_LST_doy2023134021109_aid0001.tif (0.4 MB)
  - drcs_activations/202405_Heat_TX/ECOSTRES

(150, 13289603352)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys


['drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0702.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0703.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0704.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0705.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0706.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0707.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0708.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0709.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0710.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0703.tif',
 'drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0704.tif',
 'drcs_activations/202407_Hurricane_Beryl/trop

In [11]:
def create_cog_filename_tropomi(f, EVENT_NAME):
    """Create COG filename for TROPOMI files using date from EVENT_NAME."""
    from pathlib import Path
    import re
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Extract year and month from EVENT_NAME (e.g., 202407_Hurricane_Beryl)
    event_match = re.match(r'(\d{4})(\d{2})_', EVENT_NAME)
    if not event_match:
        # Fallback if EVENT_NAME doesn't match expected pattern
        return f'{EVENT_NAME}_{filename}{extension}'
    
    year = event_match.group(1)
    month = event_match.group(2)
    
    # Parse TROPOMI filename pattern: TROPOMI_CO_0702 or TROPOMI_CH4_0703
    # The numbers at the end are MMDD (month and day)
    parts = filename.split('_')
    
    if len(parts) >= 3 and len(parts[-1]) == 4 and parts[-1].isdigit():
        # Extract the date components
        date_str = parts[-1]  # 0702, 0703, etc.
        file_month = date_str[:2]
        day = date_str[2:4]
        
        # Verify month matches EVENT_NAME month (or use the one from filename if needed)
        if file_month != month:
            # Use the month from filename if it doesn't match EVENT_NAME
            formatted_date = f"{year}-{file_month}-{day}"
        else:
            formatted_date = f"{year}-{month}-{day}"
        
        # Get the rest of the filename (everything except the date)
        remaining_parts = parts[:-1]
        
        # Build new filename
        new_parts = [EVENT_NAME] + remaining_parts + [formatted_date, 'day']
        cog_filename = '_'.join(new_parts) + extension
    else:
        # Fallback if pattern doesn't match
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename

filter_str = ''

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_tropomi(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")



Testing WM filename:
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-02_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-03_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-04_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-05_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-06_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-07_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-08_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-09_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-10_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-03_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-04_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-05_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-09_day.tif


In [12]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_tropomi, 
                                target_dir = "Sentinel-5P", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-02_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-03_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-04_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-05_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-06_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-07_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-08_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-09_day.tif
  202407_Hurricane_Beryl_TROPOMI_CO_2024-07-10_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-03_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-04_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-05_day.tif
  202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-09_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202407_Hurricane_Beryl/tropomi
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-5P

🌊 Processing Files (Chunked)
✅ Local output directory ready:

Reading input: /tmp/tmp8guntbz7_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpq15rju8v.tif


   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-5P/202407_Hurricane_Beryl_TROPOMI_CO_2024-07-02_day.tif
   [MEMORY] Final: 324.2 MB (Change: +36.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202407_Hurricane_Beryl_TROPOMI_CO_2024-07-02_day.tif

[2/13] Processing: drcs_activations/202407_Hurricane_Beryl/tropomi/carbon_monoxide/TROPOMI_CO_0703.tif
   Output filename: 202407_Hurricane_Beryl_TROPOMI_CO_2024-07-03_day.tif
   [MEMORY] Initial: 324.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache


Reading input: /tmp/tmpro3kzhtm_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmp7pn47x9h.tif


   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=2025/2025
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - No overviews found
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-5P/202407_Hurricane_Beryl_TROPOMI_CO_2024-07-03_day.tif
   [MEMORY] Final: 336.7 MB (Change: +12.5 MB)
✅ Chunked COG conversion function defined wit

...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-03_day.tif

[11/13] Processing: drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0704.tif
   Output filename: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-04_day.tif
   [MEMORY] Initial: 337.3 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=3844/3844
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for co

Reading input: /tmp/tmpbh0qubx7_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmpx6dlk9x5.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-04_day.tif

[12/13] Processing: drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0705.tif
   Output filename: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-05_day.tif
   [MEMORY] Initial: 337.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=2401/2401
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for co

Reading input: /tmp/tmpir3n3hf4_temp.tif

Updating dataset tags...
Writing output to: /tmp/tmppmhozlvx.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-05_day.tif

[13/13] Processing: drcs_activations/202407_Hurricane_Beryl/tropomi/methane/TROPOMI_CH4_0709.tif
   Output filename: 202407_Hurricane_Beryl_TROPOMI_CH4_2024-07-09_day.tif
   [MEMORY] Initial: 337.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=nan, max=nan, center sample non-zero=3481/3481
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [PREDICTOR] Data type: float64, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for co

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")